# MemWeave 2.0
### An Adaptive, Temporal and Budget-Constrained Memory Architecture for Long-Horizon Conversational Assistants
**Real-data research notebook on LongMemEval-S (cleaned)**

This notebook upgrades the first MemWeave notebook. It keeps the same benchmark and the same no-fabrication rule, but changes three things:

1. **The framing is fixed.** The first notebook gave the selector the question *before* it chose what to keep. That is query-conditioned evidence selection. This notebook tests **two settings** and keeps them separate:
   - **Setting B (primary): query-agnostic memory retention.** The system must decide what to keep under a token budget *without seeing the future question*. Only afterwards is the question asked and evidence retention measured. This is the actual long-term-memory problem.
   - **Setting A (reference): query-conditioned selection.** The question is known. This is what the first notebook did, and it is the setting where BM25 / dense retrieval are natural baselines.
2. **The algorithm now implements the original MemWeave ideas**: temporal supersession detection, rarity and importance estimation, a learned utility network, and a budget-constrained, redundancy-aware optimizer.
3. **The evaluation is paper-grade**: repeated stratified cross-validation over several seeds, paired bootstrap CIs, Wilcoxon tests with Holm correction, ablations, efficiency measurements, and per-category breakdowns.

> **Research integrity.** No benchmark data and no result numbers are written in this notebook. Every table is produced by the cells below from the downloaded benchmark. Until you have run it end-to-end in an internet-enabled environment, this notebook contains **no results**, and nothing here may be described as a finding. Whether MemWeave 2.0 beats any baseline is an empirical question this notebook is built to answer, including the possibility that it does not.

**How to run.** Google Colab (a GPU runtime is only needed for the optional dense-retrieval baseline) or a local Jupyter. Run cells top to bottom. Feature extraction is the slow step; it is cached to disk.

## 1. Research questions, hypotheses and pre-registered protocol

**RQ1 (primary, Setting B).** Under a fixed token budget, can a system that sees only the conversation history (not the future question) retain more answer-supporting sessions than recency, random, and hand-built heuristic retention policies?

**RQ2 (Setting A).** When the question *is* known, does the learned utility + redundancy-aware optimizer compete with BM25 and dense retrieval under the same budget?

**RQ3 (mechanism).** Which components contribute: temporal supersession features, rarity/uniqueness, importance cues, redundancy-aware selection, cost-aware selection, and the utility model class?

| Hypothesis | Statement |
|---|---|
| H1 | In Setting B, learned MemWeave 2.0 retains more evidence sessions than recency and random at matched budgets. |
| H2 | In Setting B, learned MemWeave 2.0 retains more evidence than the *unlearned* heuristic (equal-weight importance + rarity + novelty, with and without MMR), i.e. learning adds value beyond hand-built rules. |
| H3 | Redundancy-aware, cost-aware selection improves over plain top-score filling. |
| H4 | Temporal supersession features help most on knowledge-update and temporal-reasoning questions. |

**Fixed before looking at any result** (these are in the config cell, not tuned on test data):
- primary budget = **10%** of each history's token count; secondary budgets 5%, 20%, 30%
- primary metric = **evidence recall** (fraction of gold evidence sessions retained)
- evaluation = repeated 5-fold stratified cross-validation over 3 seeds, so every question is a test question exactly once per seed
- all hyper-parameters of the optimizer are chosen on an inner validation split of the *training* fold only, at the primary budget
- significance = paired bootstrap 95% CI on seed-averaged per-question differences, Wilcoxon signed-rank test, Holm correction across comparisons

## 2. Environment and configuration

In [ ]:
import os, sys, json, re, time, math, random, hashlib, platform, pickle, warnings, datetime, itertools
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import rankdata
import sklearn
try:
    from IPython.display import display
except Exception:
    display = print
warnings.filterwarnings("ignore")

CODE_VERSION = "memweave-2.0.0"
SMOKE = os.environ.get("MEMWEAVE_SMOKE") == "1"   # developer pipeline self-test only; leave unset for real runs

CFG = dict(
    seeds=[0, 1, 2],
    n_folds=5,
    budget_fracs=[0.05, 0.10, 0.20, 0.30],   # fraction of each history's token count
    primary_budget=0.10,                      # fixed in advance
    k_ret=[3, 5],                             # retrieval cut-offs applied AFTER memory retention (Setting B)
    max_questions=None,                       # set e.g. 100 for a quick trial run
    exact_tokens=True,                        # tiktoken cl100k if available, else whitespace
    use_dense="auto",                         # "auto" = only if a GPU and sentence-transformers are available
    dense_model="sentence-transformers/all-MiniLM-L6-v2",
    dense_chunk_words=120, dense_max_chunks=40,
    topic_sim_percentile=90,                  # supersession topic threshold (per history)
    n_boot=5000,
    run_qa_proxy=False, qa_n=100, qa_model="Qwen/Qwen2.5-1.5B-Instruct", qa_topk=3, qa_words_per_session=900,
    rebuild_cache=False,
)
if SMOKE:
    CFG.update(seeds=[0, 1], n_folds=3, n_boot=400, exact_tokens=False, use_dense=False)

OUT_DIR = Path(os.environ.get("MEMWEAVE_OUT", "memweave2_results")); OUT_DIR.mkdir(exist_ok=True)
rng_global = np.random.default_rng(12345); random.seed(12345); np.random.seed(12345)

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__)
print("SMOKE self-test mode:", SMOKE)
print(json.dumps({k: v for k, v in CFG.items()}, indent=1, default=str))

## 3. Download the real benchmark

LongMemEval-S cleaned (about 265 MB). Do **not** substitute the oracle variant; it contains only evidence sessions and makes memory selection trivial. If you already have the file, set the `MEMWEAVE_DATA` environment variable or edit `DATA_PATH`.

In [ ]:
DATA_DIR = Path("data/longmemeval"); DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = Path(os.environ.get("MEMWEAVE_DATA", DATA_DIR / "longmemeval_s_cleaned.json"))
DATA_URL = ("https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/"
            "resolve/main/longmemeval_s_cleaned.json")

if not DATA_PATH.exists():
    print("Downloading LongMemEval-S (several minutes)...")
    try:
        import urllib.request
        urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    except Exception as exc:
        if DATA_PATH.exists(): DATA_PATH.unlink()
        raise RuntimeError("Download failed. Fetch longmemeval_s_cleaned.json manually from the "
                           "Hugging Face dataset page and set DATA_PATH.") from exc
print("Dataset:", DATA_PATH, "| MB:", round(DATA_PATH.stat().st_size / 1e6, 1))

## 4. Load, audit, and leakage check

Besides the schema audit from the first notebook, this cell checks for **label leakage channels**. Benchmarks sometimes encode evidence status in identifiers or annotations. The code below never uses session IDs or the per-turn `has_answer` flag as features; the check simply tells you whether such a shortcut exists so you can state it in the paper.

In [ ]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)
if not isinstance(raw, list) or not raw:
    raise ValueError("Expected a non-empty top-level JSON list.")
if CFG["max_questions"]:
    sel = np.random.default_rng(0).permutation(len(raw))[:CFG["max_questions"]]
    raw = [raw[i] for i in sorted(sel)]
required = {"question_id", "question", "answer", "haystack_sessions", "haystack_session_ids", "answer_session_ids"}
missing = required - set(raw[0])
if missing: raise ValueError(f"Unexpected schema; missing: {missing}")
print("Questions:", len(raw), "| keys:", sorted(raw[0].keys()))
print("Has haystack_dates:", "haystack_dates" in raw[0], "| has question_date:", "question_date" in raw[0])

rows = []
for r in raw:
    ids = [str(x) for x in r["haystack_session_ids"]]; gold = {str(x) for x in (r.get("answer_session_ids") or [])}
    if len(ids) != len(r["haystack_sessions"]): raise ValueError(f"{r['question_id']}: ids/sessions length mismatch")
    rows.append(dict(qid=r["question_id"], qtype=r.get("question_type", "unknown"), n_sessions=len(ids),
                     n_gold=len(gold), missing_gold=len(gold - set(ids)),
                     gold_prefix=np.mean([g.startswith("answer_") for g in gold]) if gold else np.nan,
                     nongold_prefix=np.mean([i.startswith("answer_") for i in ids if i not in gold]) if len(ids) > len(gold) else np.nan))
audit = pd.DataFrame(rows)
display(audit.qtype.value_counts().rename("count").to_frame())
display(audit[["n_sessions", "n_gold", "missing_gold"]].describe())
if audit.missing_gold.sum() > 0: raise ValueError("Some gold session IDs are not in the haystack.")
print("Questions with NO gold evidence (excluded from recall metrics):", int((audit.n_gold == 0).sum()))

gp, ngp = audit.gold_prefix.mean(), audit.nongold_prefix.mean()
print(f"\nLEAKAGE CHECK  fraction of gold IDs starting with 'answer_': {gp:.3f} | of non-gold IDs: {ngp:.3f}")
if gp > 0.5 and ngp < 0.05:
    print("WARNING: session IDs encode evidence status in this file. They are NOT used as features anywhere in this "
          "notebook. State this in the paper and never add an ID-derived feature.")
has_flag = any(isinstance(t, dict) and "has_answer" in t for s in raw[0]["haystack_sessions"] for t in (s if isinstance(s, list) else []))
print("Per-turn 'has_answer' flags present:", has_flag, "(never used for selection; reserved for a turn-level extension)")

## 5. MemWeave 2.0 architecture and how it maps to code

```
 conversation history (sessions, in time order)
          │
          ▼
 [1] Memory Encoder ────────── session → memory unit (user/assistant text, entities, token cost)
          │
          ├─► [2] Temporal Intelligence Engine ── later-session topic similarity, entity/number conflict
          │                                        → supersession signal, "latest on topic" flag
          ├─► [3] Rarity & Importance Estimator ─ rare-term mass, uniqueness/novelty, first-person,
          │                                        numeric/entity/time-reference/update-marker density
          ▼
 [4] Adaptive Utility Network ─ learns P(session is future evidence) from the features above
          │                      (trained on training-fold evidence links only)
          ▼
 [5] Budget-Constrained Optimizer ─ greedy marginal utility per token^γ with a redundancy penalty λ
          │                          and a supersession penalty ν (all tuned on inner validation)
          ▼
 retained memory ──► [6] Future-Query Retrieval (BM25 over retained memory) ──► evidence recall@k
```

Setting B uses components 1–5 with **no query information**. Setting A adds query-conditioned features (TF-IDF, BM25, optional dense similarity) to the same network.

**Memory unit.** One complete session. This keeps the gold mapping (`answer_session_ids`) exact. Finer, turn-level units are a natural extension but are not evaluated here.

## 6. Memory Encoder, Temporal Intelligence Engine, and Rarity/Importance Estimator

All **static** features below use only the conversation history. None uses the question, the gold labels, the session IDs, or the `has_answer` flags. They are therefore legal inputs for Setting B.

| Group | Features | Purpose |
|---|---|---|
| base | recency, days-before-question, log tokens, relative length | cost and age |
| importance | user-turn ratio, user word fraction, user-turn length, first-person density, number density, entity density, time-reference density, update-marker density | personal, factual content likely to be asked about later |
| rarity | mean in-history term rarity, fraction of terms unique to the session | information hard to recover elsewhere |
| uniqueness | novelty (1 − max similarity to any other session), centrality | redundancy / duplication |
| temporal | max similarity to a later session, to an earlier session, **supersession signal**, latest-on-topic flag | knowledge updates |

Supersession signal for session *i*: topic similarity to its most similar **later** session, multiplied by how much the entity/number sets of the two sessions differ. High topic overlap with changed values suggests *i* may have been updated later.

Each feature is also added as a **within-history percentile**, because selection is a within-history ranking problem.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS as STOP

WORD_RE = re.compile(r"[A-Za-z0-9']+")
NUM_RE = re.compile(r"\b\d+(?:[.,]\d+)?\b")
SENT_RE = re.compile(r"(?<=[.!?])\s+|\n+")
DATE_RE = re.compile(r"(\d{4})[/-](\d{1,2})[/-](\d{1,2})")
FIRST_PERSON = {"i", "my", "me", "mine", "i'm", "i've", "i'd", "i'll", "myself", "we", "our", "ours"}
TEMPORAL = {"yesterday", "today", "tomorrow", "ago", "last", "next", "week", "weeks", "month", "months", "year", "years",
            "monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday", "january", "february", "march",
            "april", "may", "june", "july", "august", "september", "october", "november", "december", "recently",
            "since", "until", "tonight", "morning", "evening", "weekend"}
UPDATE = {"now", "anymore", "changed", "switched", "moved", "started", "stopped", "upgraded", "new", "recently",
          "previously", "instead", "finally", "quit", "joined", "left", "bought", "sold", "replaced", "updated", "used"}

# ---- tokenizer (exact if available, otherwise whitespace; recorded in the manifest) ----
TOKENIZER = "whitespace-approx"
def count_tokens(text): return max(1, len(str(text).split()))
if CFG["exact_tokens"]:
    try:
        import tiktoken
        _enc = tiktoken.get_encoding("cl100k_base"); _enc.encode("test")
        def count_tokens(text): return max(1, len(_enc.encode(str(text), disallowed_special=())))
        TOKENIZER = "tiktoken/cl100k_base"
    except Exception as e:
        print("tiktoken unavailable -> whitespace approximation:", type(e).__name__)
print("Token counting:", TOKENIZER)

def parse_date(s):
    m = DATE_RE.search(str(s)) if s is not None else None
    if not m: return None
    try: return datetime.date(int(m.group(1)), int(m.group(2)), int(m.group(3))).toordinal()
    except ValueError: return None

def session_turns(session):
    if isinstance(session, dict):
        session = session.get("turns") or session.get("messages") or session.get("conversation") or []
    turns = []
    for t in session:
        if isinstance(t, dict):
            turns.append((str(t.get("role", t.get("speaker", "unknown"))).lower(), str(t.get("content", t.get("text", "")))))
        else:
            turns.append(("unknown", str(t)))      # NOTE: has_answer / ids are deliberately never read
    return turns

def entities(text):
    out = []
    for sent in SENT_RE.split(text):
        for tok in sent.split()[1:]:
            t = tok.strip(".,;:!?\"'()[]{}")
            if len(t) > 1 and t[0].isupper() and t.lower() not in FIRST_PERSON:
                out.append(t.lower())
    return out

def encode_session(session):                       # [1] Memory Encoder
    turns = session_turns(session)
    user = "\n".join(c for r, c in turns if r == "user")
    full = "\n".join(f"{r}: {c}" for r, c in turns)
    return dict(full=full, user=user, n_turns=len(turns), n_user=sum(r == "user" for r, _ in turns),
                tokens=count_tokens(full), ents=set(NUM_RE.findall(user)) | set(entities(user)))

STATIC_NAMES = ["recency", "days_before_q", "log_tokens", "rel_length",
                "user_turn_ratio", "user_word_frac", "log_user_turn_len", "first_person", "number_density",
                "entity_density", "temporal_density", "update_density",
                "rarity_mean", "rare_frac",
                "novelty", "centrality",
                "max_later_sim", "max_earlier_sim", "update_signal", "is_latest_topic"]
GROUPS = {"base": STATIC_NAMES[0:4], "importance": STATIC_NAMES[4:12], "rarity": STATIC_NAMES[12:14],
          "uniqueness": STATIC_NAMES[14:16], "temporal": STATIC_NAMES[16:20]}
XS_NAMES = STATIC_NAMES + [n + "_pct" for n in STATIC_NAMES]
IDX = {n: i for i, n in enumerate(XS_NAMES)}
def pct(col): return rankdata(col) / len(col)

def static_features(units, dts, qd, S):
    n = len(units); F = {k: np.zeros(n) for k in STATIC_NAMES}
    cost = np.array([u["tokens"] for u in units], float)
    uterms = []
    for u in units:
        ws = [w.lower() for w in WORD_RE.findall(u["user"])]
        uterms.append(set(w for w in ws if w not in STOP and len(w) > 2))
    df = Counter(t for s in uterms for t in s)
    off = S[~np.eye(n, dtype=bool)]
    tau = np.percentile(off, CFG["topic_sim_percentile"]) if n > 2 else 1.0
    for i, u in enumerate(units):
        ws = [w.lower() for w in WORD_RE.findall(u["user"])]; nw = max(len(ws), 1)
        tw = max(len(WORD_RE.findall(u["full"])), 1)
        F["recency"][i] = (i + 1) / n
        F["days_before_q"][i] = math.log1p(max(qd - dts[i], 0)) if (qd is not None and dts[i] is not None) else math.log1p(n - 1 - i)
        F["log_tokens"][i] = math.log1p(cost[i]); F["rel_length"][i] = cost[i] / max(cost.max(), 1)
        F["user_turn_ratio"][i] = u["n_user"] / max(u["n_turns"], 1)
        F["user_word_frac"][i] = len(ws) / tw
        F["log_user_turn_len"][i] = math.log1p(len(ws) / max(u["n_user"], 1))
        c = Counter(ws)
        F["first_person"][i] = sum(c[w] for w in FIRST_PERSON) / nw
        F["number_density"][i] = len(NUM_RE.findall(u["user"])) / nw
        F["entity_density"][i] = len(entities(u["user"])) / nw
        F["temporal_density"][i] = sum(c[w] for w in TEMPORAL) / nw
        F["update_density"][i] = sum(c[w] for w in UPDATE) / nw
        ts = uterms[i]
        if ts:
            F["rarity_mean"][i] = float(np.mean([math.log((1 + n) / (1 + df[t])) for t in ts]))
            F["rare_frac"][i] = float(np.mean([df[t] == 1 for t in ts]))
        # [2] temporal intelligence
        later, earlier = S[i, i + 1:], S[i, :i]
        mx_l = float(later.max()) if len(later) else 0.0
        F["max_later_sim"][i] = mx_l; F["max_earlier_sim"][i] = float(earlier.max()) if len(earlier) else 0.0
        if len(later):
            j = i + 1 + int(later.argmax()); Ei, Ej = u["ents"], units[j]["ents"]
            uni = len(Ei | Ej); jac = len(Ei & Ej) / uni if uni else 1.0
            F["update_signal"][i] = mx_l * (1.0 - jac)
        F["is_latest_topic"][i] = float(mx_l < tau)
    sims = S.copy()
    F["novelty"] = 1.0 - sims.max(1) if n > 1 else np.ones(n)
    F["centrality"] = sims.sum(1) / max(n - 1, 1)
    return F

def bm25_scores(q, docs, k1=1.5, b=0.75):
    qt = {w.lower() for w in WORD_RE.findall(q)} - STOP
    ctr = [Counter(w.lower() for w in WORD_RE.findall(d)) for d in docs]
    dl = np.array([sum(c.values()) for c in ctr], float); avg = max(dl.mean(), 1.0); N = len(docs)
    s = np.zeros(N)
    for t in qt:
        dft = sum(1 for c in ctr if t in c)
        if dft == 0: continue
        idf = math.log(1 + (N - dft + 0.5) / (dft + 0.5))
        for j, c in enumerate(ctr):
            tf = c.get(t, 0)
            if tf: s[j] += idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl[j] / avg))
    return s

def make_dense_fn():
    mode = CFG["use_dense"]
    if mode in (False, "off"): return None
    try:
        import torch
        from sentence_transformers import SentenceTransformer
        if mode == "auto" and not torch.cuda.is_available():
            print("Dense baseline OFF (no GPU; set use_dense=True to force)."); return None
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        model = SentenceTransformer(CFG["dense_model"], device=dev)
    except Exception as e:
        print("Dense baseline OFF:", type(e).__name__, str(e)[:120]); return None
    size, maxc = CFG["dense_chunk_words"], CFG["dense_max_chunks"]
    def fn(q, docs):
        chunks, owner = [], []
        for j, d in enumerate(docs):
            w = d.split(); cs = [" ".join(w[i:i + size]) for i in range(0, len(w), size)][:maxc] or [""]
            chunks += cs; owner += [j] * len(cs)
        E = model.encode(chunks, batch_size=128, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
        qe = model.encode([q], convert_to_numpy=True, normalize_embeddings=True)[0]
        out = np.full(len(docs), -1.0); np.maximum.at(out, owner, E @ qe); return out
    print("Dense baseline ON:", CFG["dense_model"], "on", dev); return fn
DENSE_FN = make_dense_fn()
HAS_DENSE = DENSE_FN is not None
QNAMES = ["tfidf", "bm25"] + (["dense"] if HAS_DENSE else [])

def build_bundle(rec, idx):
    t0 = time.perf_counter()
    ids = [str(x) for x in rec["haystack_session_ids"]]
    units = [encode_session(s) for s in rec["haystack_sessions"]]
    dts = [parse_date(d) for d in (rec.get("haystack_dates") or [None] * len(ids))]
    ok = len(dts) == len(ids) and all(d is not None for d in dts)
    order = np.argsort(dts, kind="stable") if ok else np.arange(len(ids))
    ids = [ids[j] for j in order]; units = [units[j] for j in order]; dts = [dts[j] for j in order] if ok else [None] * len(ids)
    n = len(ids); qd = parse_date(rec.get("question_date"))
    gold_set = {str(x) for x in (rec.get("answer_session_ids") or [])}
    docs = [u["full"] for u in units]
    try:
        vec = TfidfVectorizer(stop_words="english", sublinear_tf=True, max_df=0.9); Xd = vec.fit_transform(docs)
    except ValueError:
        vec = TfidfVectorizer(sublinear_tf=True); Xd = vec.fit_transform(docs)
    S = (Xd @ Xd.T).toarray().astype(np.float32); np.fill_diagonal(S, 0.0)
    F = static_features(units, dts, qd, S)
    Xs = np.column_stack([F[k] for k in STATIC_NAMES] + [pct(F[k]) for k in STATIC_NAMES]).astype(np.float32)
    t_static = (time.perf_counter() - t0) * 1e3
    # query-conditioned features (Setting A and retrieval stage of Setting B)
    t1 = time.perf_counter(); q = str(rec["question"])
    tfidf = (Xd @ vec.transform([q]).T).toarray().ravel()
    bm25 = bm25_scores(q, docs); bm25n = bm25 / (bm25.max() + 1e-9)
    qcols = [tfidf, bm25n]
    if HAS_DENSE: qcols.append(DENSE_FN(q, docs))
    Xq = np.column_stack(qcols + [pct(c) for c in qcols]).astype(np.float32)
    return dict(idx=idx, qid=rec["question_id"], qtype=rec.get("question_type", "unknown"), question=q, answer=str(rec.get("answer", "")),
                n=n, ids=ids, cost=np.array([u["tokens"] for u in units], float), gold=np.array([i in gold_set for i in ids]),
                S=S, Xs=Xs, Xq=Xq, bm25=bm25, supersede=F["update_signal"].copy(),
                t_static_ms=t_static, t_query_ms=(time.perf_counter() - t1) * 1e3)

## 7. Extract features for all questions (cached)

This is the slowest cell. Features are query-independent except the `Xq` block, so each history is processed once and reused by every split, seed, method and ablation. Delete the cache file or set `rebuild_cache=True` after changing any feature code.

In [ ]:
CACHE = OUT_DIR / f"feature_cache_{CODE_VERSION}_{len(raw)}_{TOKENIZER.replace('/', '-')}_{'dense' if HAS_DENSE else 'nodense'}.pkl"
if CACHE.exists() and not CFG["rebuild_cache"]:
    with open(CACHE, "rb") as f: B = pickle.load(f)
    print("Loaded cached features:", CACHE.name, "|", len(B), "questions")
else:
    B, t0 = [], time.time()
    for i, rec in enumerate(raw):
        B.append(build_bundle(rec, i))
        if (i + 1) % 25 == 0 or i + 1 == len(raw):
            print(f"  {i + 1}/{len(raw)} questions | {time.time() - t0:.0f}s elapsed", end="\r")
    with open(CACHE, "wb") as f: pickle.dump(B, f)
    print("\nBuilt and cached:", CACHE.name)

print("Static feature columns:", len(XS_NAMES), "| query feature columns:", B[0]["Xq"].shape[1], "| query features:", QNAMES)
print("Mean sessions/history:", round(np.mean([b["n"] for b in B]), 1),
      "| mean tokens/history:", int(np.mean([b["cost"].sum() for b in B])),
      "| mean gold/question:", round(np.mean([b["gold"].sum() for b in B]), 2))
print("Mean static-feature time (ms/question):", round(np.mean([b["t_static_ms"] for b in B]), 1),
      "| query-feature time:", round(np.mean([b["t_query_ms"] for b in B]), 1))

## 8. Budget-Constrained Optimizer and metrics

Given utilities *u_j* (from any scorer), costs *c_j* and a token budget, the optimizer repeatedly adds the session with the highest

&emsp;&emsp;**max(u_j − λ·max_{s∈S} sim(j, s) − ν·supersede_j, ε) / c_j^γ**

that still fits. λ is the redundancy penalty (an MMR-style term), γ controls cost-awareness (γ = 0 is plain top-score filling, γ = 1 is utility per token), and ν penalizes sessions the temporal engine flags as likely superseded. Setting λ = γ = ν = 0 reproduces the standard "rank and fill" policy used for every non-tuned baseline.

**Metrics** (per question): *evidence recall* (share of gold sessions retained), *evidence coverage* (all gold sessions retained), fraction of budget used, and in Setting B *retrieval recall@k* when BM25 is run over the retained memory only (the realistic pipeline: retain, then answer a later question).

In [ ]:
def select(b, util, p):
    n, cost, budget = b["n"], b["cost"], p["budget"]
    lam, gam, nu = p.get("lam", 0.0), p.get("gamma", 0.0), p.get("nu", 0.0)
    sup, S = b["supersede"], b["S"]
    chosen, used = [], 0.0
    maxsim = np.zeros(n); avail = np.ones(n, bool)
    while True:
        gain = np.maximum(util - lam * maxsim - nu * sup, 1e-6)
        ratio = gain / np.power(cost, gam)
        ratio[~avail] = -np.inf; ratio[cost > budget - used] = -np.inf
        j = int(np.argmax(ratio))
        if not np.isfinite(ratio[j]): break
        chosen.append(j); used += cost[j]; avail[j] = False; maxsim = np.maximum(maxsim, S[:, j])
    return chosen, used

RET_COLS = [f"ret_r{k}" for k in CFG["k_ret"]]
def metrics(b, chosen, used):
    gold, ng = b["gold"], int(b["gold"].sum()); n = b["n"]
    sel = np.zeros(n, bool); sel[chosen] = True
    out = dict(used_frac=used / b["cost"].sum())
    if ng == 0:
        out.update(recall=np.nan, coverage=np.nan, **{c: np.nan for c in RET_COLS}); return out
    out["recall"] = float((sel & gold).sum() / ng); out["coverage"] = float(out["recall"] == 1.0)
    order = sorted(chosen, key=lambda j: -b["bm25"][j])          # retrieval over retained memory only
    for k, c in zip(CFG["k_ret"], RET_COLS): out[c] = float(gold[order[:k]].sum() / ng) if order else 0.0
    return out

def eval_method(bundles, utils, p, frac):
    rows, kept = [], []
    for b, u in zip(bundles, utils):
        t = time.perf_counter(); chosen, used = select(b, u, {**p, "budget": frac * b["cost"].sum()})
        ms = (time.perf_counter() - t) * 1e3
        rows.append(dict(qid=b["qid"], qtype=b["qtype"], select_ms=ms, **metrics(b, chosen, used))); kept.append(chosen)
    return rows, kept

def combos(grid):
    return [dict(lam=l, gamma=g, nu=n) for l, g, n in itertools.product(grid["lam"], grid["gamma"], grid["nu"])]

def tune_params(val_b, val_u, grid, frac):
    best, best_s = None, -1.0
    for p in combos(grid):                                       # first (simplest) combo wins ties
        r, _ = eval_method(val_b, val_u, p, frac)
        s = np.nanmean([x["recall"] for x in r])
        if s > best_s + 1e-12: best, best_s = p, s
    return best

GRID_FULL = dict(lam=[0.0, 0.15, 0.4], gamma=[0.0, 0.5, 1.0], nu=[0.0, 0.2])
GRID_NONE = dict(lam=[0.0], gamma=[0.0], nu=[0.0])
print("Optimizer grid size (full):", len(combos(GRID_FULL)))

## 9. Adaptive Utility Network and ablation variants

The utility model predicts whether a session will be gold evidence for the (unseen) question. It is trained **only on the training folds' evidence links**. The default class is a small MLP (the "utility network"); logistic regression and gradient boosting are included as model-class ablations. Positives are rare (a few per ~50 sessions), so the MLP trains on positively oversampled data; only the *ranking* of utilities matters for selection.

Ablations drop one feature group at a time, or switch off one optimizer term, and re-tune everything on the inner validation split.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split

class ConstUtility:                                              # fallback if a fold has a single label class
    def __init__(self, v): self.v = v
    def predict_proba(self, X): return np.column_stack([1 - np.full(len(X), self.v), np.full(len(X), self.v)])

def fit_utility(kind, X, y, seed):
    if len(np.unique(y)) < 2: return ConstUtility(float(y.mean()))
    if kind == "logreg":
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.5, class_weight="balanced"))
    elif kind == "hgb":
        m = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=200, class_weight="balanced", random_state=seed)
    elif kind == "mlp":
        pos = np.where(y == 1)[0]; rep = int(min(10, max(1, (len(y) - len(pos)) // max(len(pos), 1))))
        idx = np.concatenate([np.arange(len(y))] + [pos] * (rep - 1)); X, y = X[idx], y[idx]
        m = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1e-3, early_stopping=True,
                         validation_fraction=0.1, n_iter_no_change=10, max_iter=300, random_state=seed))
    else:
        raise ValueError(kind)
    return m.fit(X, y)

def cols_for(drop=()):
    kept = [n for g, ns in GROUPS.items() if g not in drop for n in ns]
    return [IDX[n] for n in kept] + [IDX[n + "_pct"] for n in kept]

def feat_matrix(bundles, cols, with_q):
    X = np.vstack([np.hstack([b["Xs"][:, cols]] + ([b["Xq"]] if with_q else [])) for b in bundles])
    y = np.concatenate([b["gold"] for b in bundles]).astype(int)
    return X, y

def util_list(model, bundles, cols, with_q):
    out = []
    for b in bundles:
        X = np.hstack([b["Xs"][:, cols]] + ([b["Xq"]] if with_q else []))
        out.append(model.predict_proba(X)[:, 1])
    return out

MAIN = "MemWeave-2.0 (full)"
VARIANTS = [   # name, dropped feature groups, model class, optimizer grid
    (MAIN,                         (),               "mlp",    GRID_FULL),
    ("- importance cues",          ("importance",),  "mlp",    GRID_FULL),
    ("- rarity",                   ("rarity",),      "mlp",    GRID_FULL),
    ("- uniqueness/novelty",       ("uniqueness",),  "mlp",    GRID_FULL),
    ("- temporal engine",          ("temporal",),    "mlp",    dict(GRID_FULL, nu=[0.0])),
    ("- base (recency/length)",    ("base",),        "mlp",    GRID_FULL),
    ("- redundancy penalty",       (),               "mlp",    dict(GRID_FULL, lam=[0.0])),
    ("- cost-awareness",           (),               "mlp",    dict(GRID_FULL, gamma=[0.0])),
    ("utility=logreg",             (),               "logreg", GRID_FULL),
    ("utility=gradient-boosting",  (),               "hgb",    GRID_FULL),
]
print("Ablation / model variants:", [v[0] for v in VARIANTS])

## 10. Baselines

**Setting B (query-agnostic)** — nothing here sees the question:
- `recency` (newest sessions first), `random` (seeded)
- `user_density` (rank by first-person density), `rarity_only` (rank by rarity)
- `heuristic_fill`: equal-weight average of first-person, rarity and novelty percentiles, rank-and-fill
- `heuristic_mmr`: the same heuristic with the redundancy-aware optimizer (λ, γ tuned on inner validation). This is the strong **unlearned** baseline for H2.

**Setting A (query-conditioned)** — the question is known:
- `recency`, `tfidf`, **`bm25`**, **`dense`** (if enabled), `bm25_mmr` (redundancy-aware, λ tuned)
- `MemWeave-2.0-Q`: the learned network with static + query features and the same optimizer

All non-tuned baselines use plain rank-and-fill (λ = γ = ν = 0), the standard way a retriever fills a context budget.

In [ ]:
HEUR_COLS = ["first_person_pct", "rarity_mean_pct", "novelty_pct"]
def col(name): return lambda b: b["Xs"][:, IDX[name]]
def qcol(name): return lambda b: b["Xq"][:, QNAMES.index(name)]
def heuristic(b): return np.mean([b["Xs"][:, IDX[c]] for c in HEUR_COLS], axis=0)

def run_fold(seed, fold, tr, te):
    types = [B[i]["qtype"] for i in tr]
    try: tr_in, tr_val = train_test_split(tr, test_size=0.2, random_state=seed, stratify=types)
    except ValueError: tr_in, tr_val = train_test_split(tr, test_size=0.2, random_state=seed)
    BIN, BVAL, BTE = [[B[i] for i in s] for s in (tr_in, tr_val, te)]
    rng = np.random.default_rng(1000 * seed + fold)
    rows, PF = [], CFG["primary_budget"]

    def record(setting, method, group, utils_val, utils_te, grid, kind="-"):
        p = tune_params(BVAL, utils_val, grid, PF) if len(combos(grid)) > 1 else combos(grid)[0]
        for frac in CFG["budget_fracs"]:
            r, kept = eval_method(BTE, utils_te, p, frac)
            for x, b, ch in zip(r, BTE, kept):
                x.update(seed=seed, fold=fold, setting=setting, method=method, group=group, kind=kind, budget=frac,
                         params=json.dumps({k: v for k, v in p.items()}))
                rows.append(x)
                if abs(frac - PF) < 1e-9 and seed == CFG["seeds"][0]: RETAINED[(setting, method)][b["idx"]] = ch

    def simple(setting, method, fn, grid=GRID_NONE, group="baseline"):
        record(setting, method, group, [fn(b) for b in BVAL], [fn(b) for b in BTE], grid)

    # ---------------- Setting B: query-agnostic retention ----------------
    simple("B", "recency", col("recency"))
    uv = [rng.random(b["n"]) for b in BVAL]; ut = [rng.random(b["n"]) for b in BTE]
    record("B", "random", "baseline", uv, ut, GRID_NONE)
    simple("B", "user_density", col("first_person_pct"))
    simple("B", "rarity_only", col("rarity_mean_pct"))
    simple("B", "heuristic_fill", heuristic)
    simple("B", "heuristic_mmr", heuristic, grid=dict(lam=[0.0, 0.15, 0.4], gamma=[0.0, 0.5, 1.0], nu=[0.0]))
    for name, drop, kind, grid in VARIANTS:
        cols = cols_for(drop)
        X, y = feat_matrix(BIN, cols, with_q=False)
        m = fit_utility(kind, X, y, seed)
        record("B", name, "main" if name == MAIN else "ablation", util_list(m, BVAL, cols, False), util_list(m, BTE, cols, False), grid, kind)

    # ---------------- Setting A: query-conditioned ----------------
    simple("A", "recency", col("recency"))
    simple("A", "tfidf", qcol("tfidf"))
    simple("A", "bm25", qcol("bm25"))
    if HAS_DENSE: simple("A", "dense", qcol("dense"))
    simple("A", "bm25_mmr", qcol("bm25"), grid=dict(lam=[0.0, 0.15, 0.4, 0.8], gamma=[0.0], nu=[0.0]))
    cols = cols_for(())
    X, y = feat_matrix(BIN, cols, with_q=True)
    m = fit_utility("mlp", X, y, seed)
    record("A", "MemWeave-2.0-Q", "main", util_list(m, BVAL, cols, True), util_list(m, BTE, cols, True), GRID_FULL, "mlp")
    return rows

## 11. Run the repeated cross-validation

3 seeds × 5 stratified folds. Every question is a test question exactly once per seed; the utility network and all optimizer parameters are fit/tuned without that fold's labels. The cell prints progress; expect this to take a while on a CPU (the number of model fits is roughly `seeds × folds × (variants + 1)`).

In [ ]:
RETAINED = {(s, m): {} for s in ("A", "B") for m in
            ["recency", "random", "user_density", "rarity_only", "heuristic_fill", "heuristic_mmr", "tfidf", "bm25", "dense",
             "bm25_mmr", "MemWeave-2.0-Q"] + [v[0] for v in VARIANTS]}
ALL, t0 = [], time.time()
idx_all = np.arange(len(B)); types_all = [b["qtype"] for b in B]
for seed in CFG["seeds"]:
    skf = StratifiedKFold(n_splits=CFG["n_folds"], shuffle=True, random_state=seed)
    for fold, (tr, te) in enumerate(skf.split(idx_all, types_all)):
        ALL += run_fold(seed, fold, tr, te)
        print(f"seed {seed} fold {fold + 1}/{CFG['n_folds']} done | {time.time() - t0:.0f}s", end="\r")
R = pd.DataFrame(ALL)
print("\nResult rows:", len(R), "| methods:", sorted(R.method.unique()))
R.to_csv(OUT_DIR / "all_results_raw.csv", index=False)

## 12. Main results

Per-question values are first **averaged over seeds** (so each question counts once), then averaged over questions. `seed_std` is the standard deviation across seeds of the macro-average, a direct measure of seed sensitivity. Questions without gold evidence are excluded from recall metrics.

In [ ]:
def qmeans(d, cols):
    return d.groupby(["method", "qid"])[cols].mean().reset_index()

def main_table(setting, frac, groups=None):
    d = R[(R.setting == setting) & np.isclose(R.budget, frac)]
    if groups: d = d[d.group.isin(groups)]
    cols = ["recall", "coverage", "used_frac"] + (RET_COLS if setting == "B" else [])
    tab = qmeans(d, cols).groupby("method")[cols].mean()
    sd = d.groupby(["method", "seed"]).recall.mean().groupby("method").std().rename("seed_std")
    tab = tab.join(sd).join(d.groupby("method").select_ms.mean().rename("select_ms"))
    return tab.sort_values("recall", ascending=False)

PF = CFG["primary_budget"]
print(f"SETTING B (query-agnostic retention) — primary budget {PF:.0%} of history tokens")
tabB = main_table("B", PF, groups=["baseline", "main"]); display(tabB.round(4))
print(f"\nSETTING A (query-conditioned selection) — budget {PF:.0%}")
tabA = main_table("A", PF); display(tabA.round(4))
tabB.to_csv(OUT_DIR / "table_main_B.csv"); tabA.to_csv(OUT_DIR / "table_main_A.csv")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for ax, (setting, groups, title) in zip(axes, [("B", ["baseline", "main"], "Setting B: query-agnostic retention"),
                                              ("A", ["baseline", "main"], "Setting A: query-conditioned selection")]):
    d = R[(R.setting == setting) & R.group.isin(groups)]
    for m, g in d.groupby("method"):
        c = g.groupby(["budget", "qid"]).recall.mean().groupby("budget").agg(["mean", "sem"])
        ax.errorbar(c.index * 100, c["mean"], yerr=1.96 * c["sem"], marker="o", capsize=2, label=m,
                    lw=2.4 if m.startswith("MemWeave") else 1.2)
    ax.set_xlabel("memory budget (% of history tokens)"); ax.set_ylabel("evidence recall"); ax.set_title(title)
    ax.set_ylim(0, 1); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_recall_vs_budget.png", dpi=160); plt.show()

## 13. Category analysis

Aggregate scores can hide failures on temporal reasoning or knowledge-update questions (H4). Check the per-category table before interpreting any aggregate.

In [ ]:
def cat_table(setting, frac, methods):
    d = R[(R.setting == setting) & np.isclose(R.budget, frac) & R.method.isin(methods)]
    q = qmeans(d.assign(qtype=d.qtype), ["recall"]).merge(d[["qid", "qtype"]].drop_duplicates(), on="qid")
    return q.pivot_table(index="qtype", columns="method", values="recall", aggfunc="mean")

methodsB = ["recency", "random", "heuristic_mmr", MAIN]
catB = cat_table("B", PF, methodsB); print("Setting B, evidence recall by question type:"); display(catB.round(3))
catB.plot(kind="bar", figsize=(11, 4.5)); plt.ylabel("evidence recall"); plt.title(f"Setting B by category @ {PF:.0%} budget")
plt.xticks(rotation=25, ha="right"); plt.ylim(0, 1); plt.tight_layout(); plt.savefig(OUT_DIR / "fig_category_B.png", dpi=160); plt.show()
catB.to_csv(OUT_DIR / "table_category_B.csv")

## 14. Statistical significance

Paired comparisons over the same questions: seed-averaged per-question differences, a percentile bootstrap 95% CI of the mean difference, a two-sided Wilcoxon signed-rank test, and Holm correction over the comparisons in each family. A CI that excludes 0 *and* a Holm-adjusted p < 0.05 are both reported; neither alone is treated as proof.

In [ ]:
def holm(ps):
    ps = np.asarray(ps, float); o = np.argsort(ps); adj = np.empty_like(ps); run = 0.0
    for r, i in enumerate(o):
        run = max(run, (len(ps) - r) * ps[i]); adj[i] = min(1.0, run)
    return adj

def paired(setting, frac, focus, others, n_boot=None, seed=0):
    n_boot = n_boot or CFG["n_boot"]; rng = np.random.default_rng(seed)
    d = R[(R.setting == setting) & np.isclose(R.budget, frac)]
    wide = d.groupby(["qid", "method"]).recall.mean().unstack()
    rows = []
    for o in others:
        diff = (wide[focus] - wide[o]).dropna().to_numpy()
        boots = rng.choice(diff, size=(n_boot, len(diff))).mean(1); lo, hi = np.quantile(boots, [.025, .975])
        try: p = float(stats.wilcoxon(diff).pvalue) if np.any(diff != 0) else 1.0
        except ValueError: p = 1.0
        rows.append(dict(setting=setting, budget=frac, comparison=f"{focus} - {o}", n=len(diff), mean_diff=diff.mean(),
                         ci_low=lo, ci_high=hi, wilcoxon_p=p))
    out = pd.DataFrame(rows); out["holm_p"] = holm(out.wilcoxon_p.values)
    out["ci_excludes_0"] = (out.ci_low > 0) | (out.ci_high < 0)
    return out

basesB = ["recency", "random", "user_density", "rarity_only", "heuristic_fill", "heuristic_mmr"]
basesA = [m for m in ["recency", "tfidf", "bm25", "dense", "bm25_mmr"] if m in set(R[R.setting == "A"].method)]
stat_tables = []
for frac in CFG["budget_fracs"]:
    stat_tables.append(paired("B", frac, MAIN, basesB)); stat_tables.append(paired("A", frac, "MemWeave-2.0-Q", basesA))
STATS = pd.concat(stat_tables, ignore_index=True); STATS.to_csv(OUT_DIR / "table_significance.csv", index=False)
print("Primary budget, Setting B:"); display(STATS[(STATS.setting == "B") & np.isclose(STATS.budget, PF)].round(4))
print("Primary budget, Setting A:"); display(STATS[(STATS.setting == "A") & np.isclose(STATS.budget, PF)].round(4))

## 15. Ablations, efficiency and ceilings

**Ablations** compare each variant to the full model, on the same questions, with paired bootstrap CIs. A component is only supported if removing it produces a CI that excludes 0 in the harmful direction. **Efficiency** reports tokens actually retained, selection latency, and feature-extraction cost per history. **Ceilings** show BM25 retrieval over the *entire* history with no memory limit, which no budgeted memory should be expected to beat.

In [ ]:
abl_names = [v[0] for v in VARIANTS if v[0] != MAIN]
ABL = paired("B", PF, MAIN, abl_names); ABL["mean_diff_meaning"] = "full minus ablated (positive = component helps)"
tab_abl = main_table("B", PF, groups=["main", "ablation"])[["recall", "coverage", "seed_std"]]
print("Ablations (Setting B, primary budget):"); display(tab_abl.round(4)); display(ABL[["comparison", "mean_diff", "ci_low", "ci_high", "holm_p", "ci_excludes_0"]].round(4))
tab_abl.to_csv(OUT_DIR / "table_ablation.csv"); ABL.to_csv(OUT_DIR / "table_ablation_paired.csv", index=False)

# knowledge-update / temporal categories (H4): full model vs "- temporal engine"
d = R[(R.setting == "B") & np.isclose(R.budget, PF) & R.method.isin([MAIN, "- temporal engine"])]
h4 = d.groupby(["qtype", "method"]).recall.mean().unstack(); h4["difference"] = h4[MAIN] - h4["- temporal engine"]
print("\nH4: effect of the temporal engine by question type"); display(h4.round(4)); h4.to_csv(OUT_DIR / "table_h4_temporal.csv")

# efficiency
eff = R[(R.setting == "B") & np.isclose(R.budget, PF) & R.method.isin(basesB + [MAIN])].groupby("method")[["used_frac", "select_ms"]].mean()
eff.loc["(feature extraction, per history)", "select_ms"] = np.mean([b["t_static_ms"] for b in B])
print("\nEfficiency (Setting B, primary budget):"); display(eff.round(4)); eff.to_csv(OUT_DIR / "table_efficiency.csv")

# ceilings: full-history retrieval with no memory budget
ceil = []
for b in B:
    ng = int(b["gold"].sum())
    if ng == 0: continue
    order = np.argsort(-b["bm25"], kind="stable")
    ceil.append({f"full_history_bm25_r{k}": b["gold"][order[:k]].sum() / ng for k in CFG["k_ret"]})
ceil = pd.DataFrame(ceil).mean(); print("\nCeiling: BM25 over the FULL history (no memory limit):"); display(ceil.round(4).to_frame("recall")); ceil.to_csv(OUT_DIR / "table_ceiling.csv")

## 16. Downstream answer-level proxy (optional, off by default)

Evidence retention is not answer accuracy. This optional cell closes part of the gap: for a sample of questions it takes each method's **retained memory**, retrieves the top-k sessions with BM25, and asks a small open instruction model to answer. Correctness is scored by normalized containment of the gold answer in the output.

**This is a cheap proxy, not the benchmark's official metric.** The official evaluation uses an LLM judge (see the LongMemEval repository's `evaluate_qa.py`). Use the proxy for relative comparison between memory policies, and run the official judge before making any QA-accuracy claim in a paper. Preference and abstention questions, and answers longer than 8 words, are excluded because containment is meaningless for them. Set `CFG["run_qa_proxy"] = True` and use a GPU. The generation code below was written but not run by the notebook author.

In [ ]:
def norm_text(s): return re.sub(r"[^a-z0-9 ]", "", str(s).lower()).strip()

def qa_context(b, chosen, k, words):
    rec = raw[b["idx"]]
    sess = dict(zip([str(x) for x in rec["haystack_session_ids"]], rec["haystack_sessions"]))
    dates = dict(zip([str(x) for x in rec["haystack_session_ids"]], rec.get("haystack_dates") or [""] * len(sess)))
    top = sorted(chosen, key=lambda j: -b["bm25"][j])[:k]
    parts = []
    for j in top:
        sid = b["ids"][j]; txt = "\n".join(f"{r}: {c}" for r, c in session_turns(sess[sid]))
        parts.append(f"[Session date: {dates.get(sid, '')}]\n" + " ".join(txt.split()[:words]))
    return "\n\n".join(parts)

def eligible_for_qa(b):
    return (b["gold"].sum() > 0 and not b["qid"].endswith("_abs") and "preference" not in b["qtype"]
            and len(b["answer"].split()) <= 8)

QA_RESULTS = None
if CFG["run_qa_proxy"]:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(CFG["qa_model"])
    lm = AutoModelForCausalLM.from_pretrained(CFG["qa_model"], torch_dtype=torch.float16, device_map="auto")
    elig = [b for b in B if eligible_for_qa(b)]
    samp = [elig[i] for i in np.random.default_rng(0).permutation(len(elig))[:CFG["qa_n"]]]
    qa_methods = ["recency", "random", "heuristic_mmr", MAIN]
    rows = []
    for b in samp:
        full_ch = list(np.argsort(-b["bm25"], kind="stable")[:CFG["qa_topk"]])
        sources = {m: RETAINED[("B", m)].get(b["idx"], []) for m in qa_methods}; sources["full_history_bm25"] = full_ch
        for m, ch in sources.items():
            ctx = qa_context(b, ch, CFG["qa_topk"], CFG["qa_words_per_session"])
            msgs = [{"role": "system", "content": "Answer the question using only the conversation excerpts. Reply with a short answer. If the excerpts do not contain it, reply 'unknown'."},
                    {"role": "user", "content": f"Excerpts:\n{ctx}\n\nQuestion: {b['question']}\nShort answer:"}]
            ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(lm.device)
            with torch.no_grad(): out = lm.generate(ids, max_new_tokens=48, do_sample=False)
            ans = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
            rows.append(dict(qid=b["qid"], method=m, correct=float(norm_text(b["answer"]) in norm_text(ans))))
    QA_RESULTS = pd.DataFrame(rows); QA_RESULTS.to_csv(OUT_DIR / "qa_proxy_results.csv", index=False)
    wide = QA_RESULTS.pivot(index="qid", columns="method", values="correct")
    print("QA proxy accuracy (containment):"); display(wide.mean().round(3).to_frame("accuracy"))
    rng_q = np.random.default_rng(0)
    for o in [m for m in wide.columns if m != MAIN]:
        diff = (wide[MAIN] - wide[o]).to_numpy(); bt = rng_q.choice(diff, size=(CFG["n_boot"], len(diff))).mean(1)
        print(f"{MAIN} - {o}: mean {diff.mean():+.3f}, 95% CI [{np.quantile(bt, .025):+.3f}, {np.quantile(bt, .975):+.3f}]")
else:
    print("QA proxy skipped (CFG['run_qa_proxy'] is False).")

## 17. Save artifacts and run manifest

Everything a reviewer needs to audit a claim: raw per-question rows, aggregate tables (CSV and LaTeX), figures, and a manifest with the dataset hash, configuration, library versions, tokenizer, and whether the dense baseline was enabled.

In [ ]:
def to_latex(df, name, caption):
    try:
        s = df.round(4).to_latex(caption=caption, label=f"tab:{name}", float_format="%.4f")
        (OUT_DIR / f"{name}.tex").write_text(s)
    except Exception as e:
        print("LaTeX export skipped for", name, "-", type(e).__name__)
to_latex(tabB, "table_main_B", "Setting B: query-agnostic retention at the primary budget.")
to_latex(tabA, "table_main_A", "Setting A: query-conditioned selection at the primary budget.")
to_latex(tab_abl, "table_ablation", "Ablations in Setting B at the primary budget.")
to_latex(catB, "table_category_B", "Evidence recall by question category (Setting B).")

pkgs = {}
for p in ["numpy", "pandas", "scipy", "sklearn", "matplotlib", "tiktoken", "sentence_transformers", "torch", "transformers"]:
    try: pkgs[p] = __import__(p).__version__
    except Exception: pkgs[p] = None
manifest = dict(project="MemWeave 2.0", code_version=CODE_VERSION, dataset="LongMemEval-S cleaned", dataset_url=DATA_URL,
                dataset_file=str(DATA_PATH), dataset_sha256=hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
                n_questions=len(raw), config=CFG, tokenizer=TOKENIZER, dense_baseline_enabled=HAS_DENSE,
                smoke_test_mode=SMOKE, timestamp_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),
                python=sys.version, platform=platform.platform(), packages=pkgs)
(OUT_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
print("Saved:", sorted(p.name for p in OUT_DIR.iterdir() if p.suffix in (".csv", ".tex", ".json", ".png")))

## 18. Paper scaffold (fill from the saved tables only)

Every `[RESULT]` below must be filled from a file in `memweave2_results/`. Do not write a number you cannot point to a CSV for.

**Title.** MemWeave 2.0: A Temporal, Budget-Constrained Memory Architecture for Long-Horizon Conversational Assistants

**Abstract.** (Problem: long-horizon assistants cannot store full histories, and must decide what to keep before knowing future questions.) (Method: learned utility over temporal-supersession, rarity, uniqueness and importance features with a redundancy- and cost-aware budget optimizer.) (Evaluation: LongMemEval-S, repeated 5-fold CV over 3 seeds, paired bootstrap.) (Result: `[RESULT: table_main_B.csv, table_significance.csv]`.) State clearly if the learned method does **not** beat the heuristic baselines.

**1. Introduction.** Query-agnostic retention versus query-conditioned retrieval; why the distinction matters; contributions.

**2. Related work** (verify every citation before submission): LongMemEval (Wu et al., ICLR 2025); LoCoMo (Maharana et al., 2024); MemGPT (Packer et al., 2023); Generative Agents (Park et al., 2023); MemoryBank (Zhong et al., 2023); Mem0 and A-MEM (2025); retrieval-augmented generation (Lewis et al., 2020); BM25 (Robertson and Zaragoza, 2009); Maximal Marginal Relevance (Carbonell and Goldstein, 1998); dense retrieval baselines.

**3. Problem formulation.** History of sessions H = (s₁ … s_n) with costs c_i, budget B, unknown future question q with gold evidence set G(q) ⊆ H. Choose M ⊆ H with Σ_{i∈M} c_i ≤ B to maximize E_q |M ∩ G(q)| / |G(q)|.

**4. Method.** Section 5 above. Complexity: feature extraction O(n²·V) for pairwise similarity (n sessions, vocabulary V); greedy selection O(n²) per history.

**5. Experimental protocol.** Sections 1, 8–11 above.

**6. Results.** `[RESULT: Sections 12–15 tables and figures]`.

**7. Threats to validity (state all of these).**
1. *Benchmark construction artifacts.* Evidence sessions in LongMemEval may differ stylistically from distractor sessions, so personal-content cues (first-person density, entities) could partly reflect dataset construction rather than general memory value. Report the ablation `- importance cues` and be explicit; test on a second dataset (for example LoCoMo) before claiming generality.
2. *Identifier leakage.* Check Section 4 output; IDs are never used here.
3. *Session-level units.* No fine-grained fact compression is evaluated.
4. *Retention is not answer accuracy.* Run the official LLM-judge evaluation before QA claims; Section 16 is a proxy.
5. *Supervision.* Labels are evidence-session membership for training-fold questions, a proxy for future utility under the benchmark's question distribution.
6. *Single benchmark, single history length.*
7. *Hyper-parameters tuned at one budget* (the primary one) and reused at others.

**8. Conclusion and limitations.** Report negative and null results as they are.